In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.express as px
from sklearn.manifold import TSNE
import umap

data = pd.read_csv("C:/Users/Brayan Gutierrez/Desktop/RNAseq-AMD/Dataset/aak100_cpmdat.csv")

data = data.drop('Unnamed: 0', axis=1)

data_num = data.drop('mgs_level', axis=1)
data_levels = data['mgs_level']

data_num = StandardScaler().fit_transform(data_num)
working_data = pd.DataFrame(data_num)

np.mean(data_num), np.std(data_num)

working_data

# PCA

In [ ]:
# 2 componenets
PCA_AMD = PCA(n_components=2)
experiment1 = PCA_AMD.fit_transform(working_data)

principal_AMD_df = pd.DataFrame(data=experiment1, columns=["PC1", "PC2"])
principal_AMD_df["mgs_level"] = data_levels 

explained = PCA_AMD.explained_variance_ratio_ * 100

plt.figure()
plt.figure(figsize=(10,10))
plt.xticks(fontsize=12)
plt.yticks(fontsize=14)
plt.xlabel(f"Principal Component 1 ({explained[0]:.1f}%)", fontsize=20)
plt.ylabel(f"Principal Component 2 ({explained[1]:.1f}%)", fontsize=20)
plt.title("Principal Component Analysis of AMD Dataset", fontsize=20)

targets = ["MGS1", "MGS4"]
colors = ["b", "r"]

for target, color in zip(targets, colors):
    indicesToKeep = principal_AMD_df["mgs_level"] == target
    plt.scatter(
        principal_AMD_df.loc[indicesToKeep, "PC1"],
        principal_AMD_df.loc[indicesToKeep, "PC2"],
        c=color,
        s=50,
        label=target
    )

plt.legend(prop={"size": 15})
plt.show()

In [ ]:
# 3 componenets
PCA_AMD = PCA(n_components=3)
experiment1 = PCA_AMD.fit_transform(working_data)

principal_AMD_df = pd.DataFrame(experiment1, columns=["PC1", "PC2", "PC3"])
principal_AMD_df["mgs_level"] = data_levels
principal_AMD_df = principal_AMD_df[principal_AMD_df["mgs_level"].isin(["MGS1", "MGS4"])]

explained = PCA_AMD.explained_variance_ratio_ * 100

fig = px.scatter_3d(
    principal_AMD_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="mgs_level",
    title="Principal Component Analysis of AMD Dataset",
    labels={
        "PC1": f"PC1 ({explained[0]:.1f}%)",
        "PC2": f"PC2 ({explained[1]:.1f}%)",
        "PC3": f"PC3 ({explained[2]:.1f}%)",
        "mgs_level": "MGS Level"
    },
    hover_data=["mgs_level"]
)

fig.update_traces(marker=dict(size=5))
fig.update_layout(width=900, height=800)
fig.show()

# tSNE

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, step = 5000 , learning_rate=200, random_state=42)
experiment2 = tsne.fit_transform(working_data)

tsne_AMD_df = pd.DataFrame(experiment2, columns=["tsne1", "tsne2"])
tsne_AMD_df["mgs_level"] = data_levels

print("KL divergence:", tsne.kl_divergence_)

fig = px.scatter(
    tsne_AMD_df,
    x="tsne1",
    y="tsne2",
    color="mgs_level",
    title="2 Component t-SNE Visualization of AMD Dataset",
    labels={
        "tsne1": "t-SNE Dimension 1",
        "tsne2": "t-SNE Dimension 2",
        "mgs_level": "MGS Level"
    }
)

fig.update_traces(marker=dict(size=6))
fig.update_layout(width=900, height=700)

fig.show()

# Hyperparameter Tuning

In [ ]:
perps = np.arange(5, 101, 5)
steps = np.arange(250, 5001, 250)

for perp in perps:
    for step in steps:

        tsne = TSNE(
            n_components=2,
            perplexity=perp,
            learning_rate=200,
            max_iter=step,
            random_state=42
        )

        experiment2 = tsne.fit_transform(working_data)

        tsne_AMD_df = pd.DataFrame(experiment2, columns=["tsne1", "tsne2"])
        tsne_AMD_df["mgs_level"] = data_levels

        print(f"perplexity={perp}, max_iter={step}, KL divergence={tsne.kl_divergence_}")

        fig = px.scatter(
            tsne_AMD_df,
            x="tsne1",
            y="tsne2",
            color="mgs_level",
            title=f"t-SNE: perplexity={perp}, max_iter={step}",
            labels={
                "tsne1": "t-SNE Dimension 1",
                "tsne2": "t-SNE Dimension 2",
                "mgs_level": "MGS Level"
            }
        )

        fig.update_traces(marker=dict(size=6))
        fig.update_layout(width=900, height=700)
        fig.show()

In [ ]:
tsne = TSNE(n_components=3, perplexity=30, learning_rate=200, random_state=42)

experiment2 = tsne.fit_transform(working_data)

tsne_AMD_df = pd.DataFrame(experiment2, columns=["tsne1", "tsne2", "tsne3"])
tsne_AMD_df["mgs_level"] = data_levels

print("KL divergence:", tsne.kl_divergence_)

fig = px.scatter_3d(
    tsne_AMD_df,
    x="tsne1",
    y="tsne2",
    z="tsne3",
    color="mgs_level",
    title="3 Component t-SNE Visualization of AMD Dataset",
    labels={
        "tsne1": "t-SNE Dimension 1",
        "tsne2": "t-SNE Dimension 2",
        "tsne3": "t-SNE Dimension 3",
        "mgs_level": "MGS Level"
    }
)

fig.update_traces(marker=dict(size=4))
fig.update_layout(width=900, height=800)

fig.show()

# UMAP

In [ ]:
umap_2d = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
)

experiment_umap_2d = umap_2d.fit_transform(working_data)

umap_AMD_df_2d = pd.DataFrame(experiment_umap_2d, columns=["UMAP1", "UMAP2"])
umap_AMD_df_2d["mgs_level"] = data_levels

fig = px.scatter(
    umap_AMD_df_2d,
    x="UMAP1",
    y="UMAP2",
    color="mgs_level",
    title="2 Componenet UMAP Visualization of AMD Dataset",
    labels={
        "UMAP1": "UMAP Dimension 1",
        "UMAP2": "UMAP Dimension 2",
        "mgs_level": "MGS Level"
    }
)

fig.update_traces(marker=dict(size=6))
fig.update_layout(width=900, height=700)

fig.show()

# Hyperparameter Tuning

In [ ]:
neigh = np.arange(2, 50, 2)
dist = np.arange(0.0, 0.9, 0.1)

for neighb in neigh:
    for dista in dist:


        umap_2d = umap.UMAP(
            n_components=2,
            n_neighbors=neighb,
            min_dist=dista,
            random_state=42
        )

        experiment_umap_2d = umap_2d.fit_transform(working_data)

        umap_AMD_df_2d = pd.DataFrame(experiment_umap_2d, columns=["UMAP1", "UMAP2"])
        umap_AMD_df_2d["mgs_level"] = data_levels

        fig = px.scatter(
            umap_AMD_df_2d,
            x="UMAP1",
            y="UMAP2",
            color="mgs_level",
            title="2 Componenet UMAP Visualization of AMD Dataset",
            labels={
                "UMAP1": "UMAP Dimension 1",
                "UMAP2": "UMAP Dimension 2",
                "mgs_level": "MGS Level"
            }
        )

        fig.update_traces(marker=dict(size=6))
        fig.update_layout(width=900, height=700)

        fig.show()

In [ ]:
umap_3d = umap.UMAP(
    n_components=3,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
)

experiment_umap_3d = umap_3d.fit_transform(working_data)

umap_AMD_df_3d = pd.DataFrame(experiment_umap_3d, columns=["UMAP1", "UMAP2", "UMAP3"])
umap_AMD_df_3d["mgs_level"] = data_levels

fig = px.scatter_3d(
    umap_AMD_df_3d,
    x="UMAP1",
    y="UMAP2",
    z="UMAP3",
    color="mgs_level",
    title="3 Componenet UMAP Visualization of AMD Dataset",
    labels={
        "UMAP1": "UMAP Dimension 1",
        "UMAP2": "UMAP Dimension 2",
        "UMAP3": "UMAP Dimension 3",
        "mgs_level": "MGS Level"
    }
)

fig.update_traces(marker=dict(size=4))
fig.update_layout(width=900, height=800)

fig.show()